# Task 6.1 Estimator 옵션 설정하기

**Overview:** 이 노트북은 실행 설정과 오류 완화 설정을 컨트롤하는 estimator 옵션에 대해 다룹니다.

In [1]:
# Setup: Import necessary libraries
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator, EstimatorOptions as Options
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.options import MeasureNoiseLearningOptions,LayerNoiseLearningOptions,ZneOptions

print("Libraries imported successfully.")

Libraries imported successfully.


## Objective 1: EstimatorOptions

EstimatorV2의 `Options` 클래스는 `Estimator` 실행에 적용되는 여러 가지 환경설정을 제공합니다. 이러한 설정은 양자 계산의 속도나 정확도, 리소스 사용량에도 영향을 줍니다.

### 속성


| 속성 | 설명 | 참고 |
|-----------|-------------|----------------|
| **`_VERSION`** | 내부 버전 트래킹 |  |
| **`max_execution_time`** | job의 최대 실행 시간 (초 단위) |  |
| **`environment`** | 실행 환경 메타데이터 | `EnvironmentOptions` 오브젝트 |
| **`simulator`** | 시뮬레이터 전용 옵션 | `{'noise_model': None, 'seed_simulator': None}` |
| **`default_precision`** | 기댓값의 목표 정확도 (기본값) |  Float, 기본값은 `0.015625` (1/√4096)  |
| **`default_shots`** | 회로의 실행 횟수 (기본값) | Integer, 기본값은 없으며, `default_precision` 값이 우선 적용됨 |
| **`resilience_level`** | 오류 완화 레벨 | `0` (없음) ~ `2` (중간) 까지 지원 |
| **`seed_estimator`** | 결과를 재생산하기 위해 적용할 수 있는 시드값 | Integer |
| **`dynamical_decoupling`** | dynamical decoupling을 활성화 및 설정 | `DynamicalDecouplingOptions` 오브젝트 |
| **`resilience`** | 고급 오류 완화 설정 | `Options.ResilienceOptionsV2` 오브젝트 |
| **`execution`** | 실행 관련 설정 | `ExecutionOptionsV2 ` 오브젝트 |
| **`twirling`** | 게이트 twirling 설정 | `Options.TwirlingOptions` 오브젝트 |
| **`experimental`** | 실험적 기능 | 실험적 기능의 dictionary |

In [2]:
# Example: Configuring Estimator Options

# Initialize service (assuming credentials is already saved)
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a simple test circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
observable = SparsePauliOp("ZZ")

# Create Options object with custom settings
options = Options()

# Set basic execution parameters
options.max_execution_time = 300
options.default_shots = 2048 
options.default_precision = 0.01
options.seed_estimator = 12345

# Set resilience level (0=none, 1=basic, 2=moderate, 3=maximum)
options.resilience_level = 2  # moderate error mitigation

print(f"Max execution time: {options.max_execution_time} seconds")
print(f"Default shots: {options.default_shots}")
print(f"Default precision: {options.default_precision}")
print(f"Resilience level: {options.resilience_level}\n")

# Create Estimator with custom options
estimator = Estimator(mode=backend, options=options)

print("Updating Options\n")

# Create initial options
initial_options = Options()
initial_options.default_shots = 1024
initial_options.max_execution_time = 300

print(f"Initial shots: {initial_options.default_shots}")
print(f"Initial max time: {initial_options.max_execution_time}")

# Update options with new values
initial_options.update(default_shots=4096, max_execution_time=900)

print(f"Updated shots: {initial_options.default_shots}")
print(f"Updated max time: {initial_options.max_execution_time}\n")

Max execution time: 300 seconds
Default shots: 2048
Default precision: 0.01
Resilience level: 2

Updating Options

Initial shots: 1024
Initial max time: 300
Updated shots: 4096
Updated max time: 900



### 메소드


#### update

여러 설정을 한 번에 업데이트합니다. 이것은 Estimator 인스턴스를 생성한 후에 여러 옵션을 변경해야 할 때 유용합니다.

In [3]:
options = Options()
options.update(
    default_shots=4096,
    max_execution_time=1200,
    resilience_level=2
)

## Objective 2: TwirlingOptions

Twirling (또는 랜덤 컴파일링) 은 회로에서 실행되는 게이트들을 랜덤하게 기능이 동일한 다른 게이트들로 바꿔 실행하여 오류의 편향성을 줄이고 파울리 노이즈 모델로 더 쉽게 노이즈를 학습할 수 있도록 하는 테크닉입니다.

| 속성 | 설명 | 참고 |
|-----------|-------------|----------------|
| **`enable_gates`** | 2-큐빗 클리포드 게이트에 대한 twirling의 적용 여부 | bool, 기본값은 `False` |
| **`enable_measure`** | 측정 연산에 대한 twirling의 적용 여부 | 기본값으로 Estimator에서는 `True`, Sampler에서는 `False` |
| **`num_randomizations`** | Twirling에서 사용할 랜덤 회로의 가짓수 | Integer 또는 'auto' |
| **`shots_per_randomization`** | 각 랜덤 회로를 실행할 shot 수 | Integer 또는 'auto' |
| **`strategy`** | Twirling 방법에 대한 옵션 | `active`, `active-circuit`, `active-accum`, `all`, 기본값은 `active-accum`|


## Objective 3: ResilienceOptions

오류 완화 설정은 `EstimatorV2`에서 사용될 오류 완화 테크닉을 컨트롤합니다. 이 설정은 노이즈가 있는 양자컴퓨터에서 컴퓨팅의 정확도를 향상시키는데 도움을 줍니다.


### 속성


| 속성 | 설명 | 참고 |
|-----------|-------------|---------|
| **`measure_mitigation`** | 측정 오류 완화의 적용 여부 | 기본값은 `True` |
| **`measure_noise_learning`** | 측정 연산의 노이즈 학습에 관한 옵션 | `MeasureNoiseLearningOptions` 오브젝트 |
| **`zne_mitigation`** | Zero-Noise Extrapolation (무소음 추정법) 적용 여부 | 기본값은 `False` |
| **`zne`** | ZNE 관련 설정 | `ZneOptions` 오브젝트 |
| **`pec_mitigation`** | Probabilistic Error Cancellation (확률적 오류 상쇄법) 적용 여부 | 기본값은 `False` |
| **`pec`** | PEC 관련 설정 | `PecOptions` 오브젝트 |
| **`layer_noise_learning`** | layer 노이즈 학습에 관한 설정  |  `LayerNoiseLearningOptions` 오브젝트 |
| **`layer_noise_model`** | 별도로 설정한 layer noise model |  `NoiseLearnerResult` 또는 `Sequence[LayerError]`  |


In [4]:
# Example: Configuring Resilience Options

# Method 1: Using resilience_level
levels = [0, 1, 2]
descriptions = [
    "No error mitigation",
    "Minimal measurement error mitigation",
    "Medium mitigation"
]

for level, desc in zip(levels, descriptions):
    options = Options()
    options.resilience_level = level
    print(f"Level {level}: {desc}")
    
# Method 2: Control with Resilience Options

options = Options()

# Configure individual resilience techniques
# Basic readout correction
options.resilience.measure_mitigation = True
# Learn measurement noise
options.resilience.measure_noise_learning = MeasureNoiseLearningOptions()
# Enable Zero-Noise Extrapolation
options.resilience.zne_mitigation = True
# Disable PEC (computationally expensive)
options.resilience.pec_mitigation = False
# Learn layer-dependent noise
options.resilience.layer_noise_learning = LayerNoiseLearningOptions() 

# Configure ZNE specifically
options.resilience.zne = ZneOptions(noise_factors=[1.0, 2.0, 3.0], extrapolator="exponential")

print("configured resilience settings:")
print(f"  Measurement mitigation: {options.resilience.measure_mitigation}")
print(f"  ZNE mitigation: {options.resilience.zne_mitigation}")
print(f"  PEC mitigation: {options.resilience.pec_mitigation}")
print(f"  ZNE noise factors: {options.resilience.zne.noise_factors}")
print(f"  ZNE extrapolator: {options.resilience.zne.extrapolator}\n")


# Phase 1: minimal resilience
quick_test = Options()
quick_test.resilience_level = 0
quick_test.default_shots = 256
print(f"  Resilience: {quick_test.resilience_level} (none)")
print(f"  Shots: {quick_test.default_shots}")

# Phase 2: moderate resilience
moderate = Options()
moderate.resilience_level = 1
moderate.default_shots = 1024
print(f"  Resilience: {moderate.resilience_level} (basic)")
print(f"  Shots: {moderate.default_shots}")

# Phase 3: High resilience
high = Options()
high.resilience_level = 2
high.default_shots = 4096
print(f"  Resilience: {high.resilience_level} (advanced)")
print(f"  Shots: {high.default_shots}")

Level 0: No error mitigation
Level 1: Minimal measurement error mitigation
Level 2: Medium mitigation
configured resilience settings:
  Measurement mitigation: True
  ZNE mitigation: True
  PEC mitigation: False
  ZNE noise factors: [1.0, 2.0, 3.0]
  ZNE extrapolator: exponential

  Resilience: 0 (none)
  Shots: 256
  Resilience: 1 (basic)
  Shots: 1024
  Resilience: 2 (advanced)
  Shots: 4096


## Objective 4: ZneOptions

Zero-Noise Extrapolation (ZNE, 무소음 추정법) 은 다음과 같이 작동하는 오류 완화 기술입니다:
1. 노이즈를 증폭하여 회로를 실행합니다.
2. 여러 노이즈 레벨에서 회로를 실행하고 결과를 측정합니다.
3. 노이즈가 없을 때의 값을 추정합니다.

### 속성


| 속성 | 설명 | 옵션 |
|-----------|-------------|---------|
| **`amplifier`** | 노이즈 증폭 방법 | `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea` |
| **`noise_factors`** | 노이즈 증폭 레벨 | Float의 리스트 (e.g., `[1.0, 2.0, 3.0]`) |
| **`extrapolator`** | 추정 방법 | `linear`, `exponential`, `double_exponential`, `polynomial_degree_1`, `polynomial_degree_2`, `polynomial_degree_3`, `polynomial_degree_4`, `polynomial_degree_5`, `polynomial_degree_6`, `polynomial_degree_7`, `fallback` |
| **`extrapolated_noise_factors`** | 추정하고자 하는 노이즈 레벨 | noise_factors로부터 얻어짐 |

In [5]:
# Example: Configuring ZNE Options

print("=== Zero-Noise Extrapolation (ZNE) Configuration ===\n")

# First, enable ZNE at the resilience level
options = Options()
options.resilience.zne_mitigation = True

# Configure ZNE parameters
print("ZNE Configuration Examples:\n")

# Basic ZNE with gate folding
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.noise_factors = [1.0, 2.0, 3.0]
options.resilience.zne.extrapolator = "linear"

print(f"Case 1:")
print(f"  Amplifier: {options.resilience.zne.amplifier}")
print(f"  Noise factors: {options.resilience.zne.noise_factors}")
print(f"  Extrapolator: {options.resilience.zne.extrapolator}")

# Advanced ZNE for better accuracy
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.noise_factors = [1.0, 1.5, 2.0, 3.0, 4.0]
options.resilience.zne.extrapolator = "exponential"

print(f"Case 2:")
print(f"  Amplifier: {options.resilience.zne.amplifier}")
print(f"  Noise factors: {options.resilience.zne.noise_factors}")
print(f"  Extrapolator: {options.resilience.zne.extrapolator}")

# Create ZNE configuration
zne_config = Options()

# Enable and configure ZNE
zne_config.resilience.zne_mitigation = True
zne_config.resilience.zne.amplifier = "gate_folding"
zne_config.resilience.zne.noise_factors = [1.0, 2.0, 3.0, 4.0]
zne_config.resilience.zne.extrapolator = "polynomial_degree_1"  # Polynomial extrapolation


# Set shots and execution time for ZNE
zne_config.default_shots = 2048
zne_config.max_execution_time = 600  # 10 minutes

print("Complete ZNE Configuration:")
print(f"  ZNE enabled: {zne_config.resilience.zne_mitigation}")
print(f"  Amplifier: {zne_config.resilience.zne.amplifier}")
print(f"  Noise factors: {zne_config.resilience.zne.noise_factors}")
print(f"  Extrapolator: {zne_config.resilience.zne.extrapolator}")
print(f"  Shots per factor: {zne_config.default_shots}")
print(f"  Total estimated shots: {zne_config.default_shots * len(zne_config.resilience.zne.noise_factors)}")
print(f"  Max execution time: {zne_config.max_execution_time} seconds")

=== Zero-Noise Extrapolation (ZNE) Configuration ===

ZNE Configuration Examples:

Case 1:
  Amplifier: gate_folding
  Noise factors: [1.0, 2.0, 3.0]
  Extrapolator: linear
Case 2:
  Amplifier: gate_folding
  Noise factors: [1.0, 1.5, 2.0, 3.0, 4.0]
  Extrapolator: exponential
Complete ZNE Configuration:
  ZNE enabled: True
  Amplifier: gate_folding
  Noise factors: [1.0, 2.0, 3.0, 4.0]
  Extrapolator: polynomial_degree_1
  Shots per factor: 2048
  Total estimated shots: 8192
  Max execution time: 600 seconds


### 참고!

Estimator의 각 `resilience_level`에서 적용되는 오류 완화 설정은 다음과 같습니다:
| Resilience Level | 정의 | 설정 |
|---|---|---|
| 0 | 오류 완화를 적용하지 않음 |  |
| 1 [기본값] | 최소한의 오류 완화만 적용함: 측정 오류 완화만 적용함 | Twirled Readout Error eXtinction (TREX) 및 측정 연산 twirling |
| 2 | 중간 레벨의 오류 완화: 일반적으로 오류를 꽤 완화해주기는 하나, 완전히 오류가 없어지는 것은 아님 | Level 1 + ZNE 및 게이트 twirling |


---
## 요약
---

이 노트북에서는 다음의 내용을 다루었습니다:

## Estimator 옵션 설정:

1. **EstimatorOptions** default_shots, default_precision, resilience_level과 같은 주요 설정이 포함됨.
2. **TwirlingOptions** 오류의 편향성을 줄이고 파울리 노이즈 모델로 노이즈 학습이 용이하게 함.
3. **ResilienceOptions** 주어진 resilience level에 따라 다양한 오류 완화 기술을 적용함.
4. **ZneOptions** 회로를 노이즈를 증폭하여 실행한 후에 노이즈가 없을 때의 값을 추정함.


---

## 연습 문제

**1) What is the primary purpose of setting resilience_level in EstimatorOptions?**

A) To control circuit execution on the backend.

B) To select the transpilation resilience optimization level.

C) To enable and coordinate error mitigation techniques.

D) To configure batching and session behavior.

**정답:**
<details> <br/>

C) `resilience_level`은 어떤 오류 완화 설정을 적용할지 (twirling, ZNE, ...) 결정하고 적용 방법을 자동으로 설정합니다.

</details>

---

**2) Which error mitigation technique is enabled at resilience level 2?**

A) Twirled Readout Error Extinction

B) Zero-Noise Extrapolation

C) Probabilistic Error Cancellation

D) Dynamic Decoupling

**정답:**
<details> <br/>

B) resilience level 2에서 활성화되는 오류 완화 설정은 Zero-Noise Extrapolation (ZNE, 무소음 추정법) 입니다. A의 TREX는 resilience level 1부터 활성화되는 설정입니다.

</details>

---

**3) Complete this code to configure ZNE with 5 noise factors and polynomial extrapolation:**

```
from qiskit_ibm_runtime import EstimatorOptions as Options

options = Options()

# Enable ZNE mitigation
options.resilience.______ = True

# Configure ZNE parameters
options.resilience.zne.amplifier = "gate_folding"
options.resilience.zne.______ = [1.0, 1.5, 2.0, 3.0, 4.0]
options.resilience.zne.extrapolator = "______"
```

Which option correctly fills all three blanks?

A) zne_enable, noise_levels, polynomial

B) zne, factors, polynomial_degree_2

C) zne_mitigation, noise_factors, polynomial_degree_2

D) enable_zne, noise_factors, poly_2

**정답:**
<details> <br/>

C) 유효한 설정 이름은 `zne_mitigation`, `noise_factors`, `polynomial_degree_2` 입니다.
</details>

----